# Data Cleaning

This notebook consolidates patch quality analysis from notebooks 05 and 19 into
an actionable cleaning workflow. It computes combined quality scores, classifies
patches into reject/review/accept tiers, and produces exclusion lists that the
training pipeline can consume directly.

The `data_cleaning` module handles the metric computation and scoring. This
notebook focuses on exploring the score distributions, calibrating the tier
thresholds, doing visual triage of borderline patches, and exporting the results.

In [ ]:
import sys
sys.path.insert(0, '../..')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_cleaning import (
    compute_all_metrics, compute_metrics_for_npz, load_patch_by_uid,
    load_quality_df, save_exclusion_list, load_exclusion_list,
    PATCH_METRIC_COLS, MASK_METRIC_COLS,
)
from data_cleaning.scoring import (
    compute_patch_quality_score, compute_mask_quality_score,
    compute_combined_score, classify_tiers, ALL_METRIC_COLS,
)
from data_cleaning.visualization import (
    gallery, plot_metric_distributions, plot_score_distribution,
    plot_patch_with_mask,
)
from data_cleaning.file_analysis import (
    aggregate_file_quality, identify_bad_files, plot_file_quality_bar,
)

PATCH_DIR = Path('/Users/axelhagen/Master/Datasets/patch_cache')
PATCH_CSV = Path('/Users/axelhagen/Master/Datasets/patch_quality_metrics.csv')
MASK_CSV = Path('/Users/axelhagen/Master/Datasets/mask_quality_metrics.csv')
COMBINED_CSV = Path('/Users/axelhagen/Master/Datasets/combined_quality.csv')
EXCLUSION_JSON = Path('/Users/axelhagen/Master/Datasets/excluded_patches.json')
FILE_EXCLUSION_JSON = Path('/Users/axelhagen/Master/Datasets/excluded_files.json')

## Load and merge existing metrics

The metric CSVs were computed in notebooks 05 and 19. If they exist, load and
merge them on uid. Otherwise, recompute from the NPZ files.

In [ ]:
if PATCH_CSV.exists() and MASK_CSV.exists():
    df_patch = pd.read_csv(PATCH_CSV)
    df_mask = pd.read_csv(MASK_CSV)
    
    # Merge on uid, keeping id columns from patch df
    id_cols = ['uid', 'npz_file', 'patch_idx', 'source_zarr', 'height']
    mask_metric_only = [c for c in df_mask.columns if c not in id_cols]
    df = df_patch.merge(df_mask[['uid'] + mask_metric_only], on='uid', how='inner')
    print(f'Loaded and merged {len(df)} patches from existing CSVs')
else:
    from tqdm.notebook import tqdm
    print('No existing CSVs found. Recomputing all metrics from NPZ files...')
    npz_files = sorted(PATCH_DIR.glob('*.npz'))
    records = []
    for npz_path in tqdm(npz_files, desc='Computing metrics'):
        records.extend(compute_metrics_for_npz(npz_path))
    df = pd.DataFrame(records)
    id_cols = ['uid', 'npz_file', 'patch_idx', 'source_zarr', 'height']
    metric_cols = [c for c in df.columns if c not in id_cols]
    df = df[id_cols + metric_cols]
    print(f'Computed metrics for {len(df)} patches')

df.head()

## Compute composite scores

In [ ]:
df['patch_quality_score'] = compute_patch_quality_score(df)
df['mask_quality_score'] = compute_mask_quality_score(df)
df['combined_score'] = compute_combined_score(df)

print(f'Combined score: min={df["combined_score"].min():.3f}, '
      f'median={df["combined_score"].median():.3f}, '
      f'max={df["combined_score"].max():.3f}')

## Score distributions

Histograms of the three composite scores. The dashed lines show the proposed
tier boundaries. Adjust `REJECT_THRESHOLD` and `ACCEPT_THRESHOLD` below based
on where the natural breakpoints fall and what the patches near each boundary
look like.

In [ ]:
REJECT_THRESHOLD = 0.20
ACCEPT_THRESHOLD = 0.60

for col in ['patch_quality_score', 'mask_quality_score', 'combined_score']:
    plot_score_distribution(df, col, REJECT_THRESHOLD, ACCEPT_THRESHOLD)

## Individual metric distributions

In [ ]:
plot_metric_distributions(df, PATCH_METRIC_COLS)
plot_metric_distributions(df, MASK_METRIC_COLS)

## Tier classification

In [ ]:
df['tier'] = classify_tiers(df, REJECT_THRESHOLD, ACCEPT_THRESHOLD)

tier_counts = df['tier'].value_counts()
print('Tier distribution:')
for tier in ['reject', 'review', 'accept']:
    n = tier_counts.get(tier, 0)
    pct = 100 * n / len(df)
    print(f'  {tier:>7s}: {n:>5d} ({pct:.1f}%)')

## Auto-reject gallery

The worst 20 patches by combined score. These should all be visually bad —
if some look acceptable, the reject threshold is too aggressive.

In [ ]:
rejected = df[df['tier'] == 'reject'].nsmallest(20, 'combined_score')
gallery(rejected['uid'].tolist(),
        f'Auto-rejected (worst 20 of {(df["tier"]=="reject").sum()})',
        df=df, patch_dir=PATCH_DIR, score_col='combined_score')

## Auto-accept gallery

The best 20 patches. These should all look clean. If some have bad masks,
the accept threshold is too generous.

In [ ]:
accepted = df[df['tier'] == 'accept'].nlargest(20, 'combined_score')
gallery(accepted['uid'].tolist(),
        f'Auto-accepted (best 20 of {(df["tier"]=="accept").sum()})',
        df=df, patch_dir=PATCH_DIR, score_col='combined_score')

## Threshold boundary check

Patches right around each threshold boundary. If the transition from
"bad" to "okay" doesn't line up with the threshold, adjust the values above.

In [ ]:
# Patches just above the reject threshold
near_reject = df[
    (df['combined_score'] >= REJECT_THRESHOLD) &
    (df['combined_score'] < REJECT_THRESHOLD + 0.05)
].nsmallest(20, 'combined_score')
gallery(near_reject['uid'].tolist(),
        f'Just above reject threshold ({REJECT_THRESHOLD})',
        df=df, patch_dir=PATCH_DIR, score_col='combined_score')

# Patches just below the accept threshold
near_accept = df[
    (df['combined_score'] >= ACCEPT_THRESHOLD - 0.05) &
    (df['combined_score'] < ACCEPT_THRESHOLD)
].nlargest(20, 'combined_score')
gallery(near_accept['uid'].tolist(),
        f'Just below accept threshold ({ACCEPT_THRESHOLD})',
        df=df, patch_dir=PATCH_DIR, score_col='combined_score')

## File-level analysis

Aggregate quality per source file. Entire files with consistently bad patches
can be excluded at the file level rather than patch-by-patch.

In [ ]:
file_df = aggregate_file_quality(df)
plot_file_quality_bar(file_df)

FILE_EXCLUDE_THRESHOLD = 0.30
bad_files = identify_bad_files(file_df, FILE_EXCLUDE_THRESHOLD)
if bad_files:
    print(f'\nFiles below threshold ({FILE_EXCLUDE_THRESHOLD}):')
    for name in bad_files:
        row = file_df.loc[name]
        print(f'  {name}: mean_score={row["mean_score"]:.3f}, '
              f'n_patches={int(row["n_patches"])}')
else:
    print(f'No files below threshold ({FILE_EXCLUDE_THRESHOLD})')

## Borderline review — paginated browser

Patches in the "review" tier, sorted worst-first. Change `PAGE` and re-run
to browse. Add uids of bad patches to `manual_exclude` below.

In [ ]:
review_df = df[df['tier'] == 'review'].sort_values('combined_score').reset_index(drop=True)
PER_PAGE = 20
total_pages = int(np.ceil(len(review_df) / PER_PAGE))
print(f'{len(review_df)} patches in review tier, {total_pages} pages')

In [ ]:
PAGE = 0

start = PAGE * PER_PAGE
end = min(start + PER_PAGE, len(review_df))
page_df = review_df.iloc[start:end]

print(f'Page {PAGE + 1}/{total_pages}  |  patches {start}-{end - 1}  |  '
      f'score range: [{page_df["combined_score"].min():.3f}, '
      f'{page_df["combined_score"].max():.3f}]')

gallery(page_df['uid'].tolist(),
        f'Review tier — page {PAGE + 1}/{total_pages}',
        df=df, patch_dir=PATCH_DIR, score_col='combined_score')

## Build exclusion list

Combine auto-rejected patches, manually flagged patches, and file-level exclusions.

In [ ]:
# Paste uids of bad patches found during manual review
manual_exclude = [
    # 'D20240131-T092044_721e57f964fb::3',
]

# Auto-rejected by tier
auto_reject_uids = set(df.loc[df['tier'] == 'reject', 'uid'])

# File-level exclusion — all patches from bad files
file_exclude_uids = set()
if bad_files:
    file_exclude_uids = set(df.loc[df['source_zarr'].isin(bad_files), 'uid'])

all_excluded = sorted(auto_reject_uids | set(manual_exclude) | file_exclude_uids)

print(f'Auto-rejected patches:   {len(auto_reject_uids)}')
print(f'Manual exclusions:       {len(manual_exclude)}')
print(f'File-level exclusions:   {len(file_exclude_uids)} (from {len(bad_files)} files)')
print(f'Total excluded (union):  {len(all_excluded)}')
print(f'Remaining for training:  {len(df) - len(all_excluded)}')

## Export

In [ ]:
# Save combined quality CSV
df.to_csv(COMBINED_CSV, index=False)
print(f'Saved {len(df)} rows to {COMBINED_CSV}')

# Save patch exclusion list
save_exclusion_list(all_excluded, EXCLUSION_JSON)
print(f'Saved {len(all_excluded)} excluded uids to {EXCLUSION_JSON}')

# Save file exclusion list
if bad_files:
    import json
    with open(FILE_EXCLUSION_JSON, 'w') as f:
        json.dump({'excluded_files': bad_files}, f, indent=2)
    print(f'Saved {len(bad_files)} excluded files to {FILE_EXCLUSION_JSON}')